In [1]:
import tensorflow as tf
import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input


In [2]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATASET_PATH = "D:/DEBI/mask/Dataset"

In [3]:
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # 🔥 IMPORTANT (NOT rescale)
    validation_split=0.2,

    # moderate augmentation (balanced, not extreme)
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

train_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

val_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

print("Classes:", train_data.class_indices)

Found 7188 images belonging to 3 classes.
Found 1794 images belonging to 3 classes.
Classes: {'mask_weared_incorrect': 0, 'with_mask': 1, 'without_mask': 2}


In [4]:
base_model = EfficientNetB0(
    weights="imagenet",   # 🔥 KEY FOR HIGH ACCURACY
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False  # freeze first

In [5]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(512, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)

x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)

output = layers.Dense(3, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

In [6]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [7]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 rescaling (Rescaling)       (None, 224, 224, 3)          0         ['input_1[0][0]']             
                                                                                                  
 normalization (Normalizati  (None, 224, 224, 3)          7         ['rescaling[0][0]']           
 on)                                                                                              
                                                                                                  
 rescaling_1 (Rescaling)     (None, 224, 224, 3)          0         ['normalization[0][0]']   

In [8]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10


225/225 [==============================] - 189s 825ms/step - loss: 0.2128 - accuracy: 0.9267 - val_loss: 0.1001 - val_accuracy: 0.9643
Epoch 2/10
225/225 [==============================] - 92s 407ms/step - loss: 0.1142 - accuracy: 0.9599 - val_loss: 0.1093 - val_accuracy: 0.9654
Epoch 3/10
225/225 [==============================] - 93s 414ms/step - loss: 0.0993 - accuracy: 0.9659 - val_loss: 0.0731 - val_accuracy: 0.9771
Epoch 4/10
225/225 [==============================] - 93s 414ms/step - loss: 0.0779 - accuracy: 0.9750 - val_loss: 0.0652 - val_accuracy: 0.9805
Epoch 5/10
225/225 [==============================] - 93s 415ms/step - loss: 0.0747 - accuracy: 0.9733 - val_loss: 0.0597 - val_accuracy: 0.9805
Epoch 6/10
225/225 [==============================] - 94s 417ms/step - loss: 0.0654 - accuracy: 0.9765 - val_loss: 0.0508 - val_accuracy: 0.9844
Epoch 7/10
225/225 [==============================] - 93s 415ms/step - loss: 0.0616 - accuracy: 0.9779 - val_loss: 0.0650 - val

In [9]:
model.save("mask_efficientnet_high_accuracy.h5")

c:\Users\SEIF\anaconda3\envs\mask_env\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [10]:
print("Class Mapping:", train_data.class_indices)

Class Mapping: {'mask_weared_incorrect': 0, 'with_mask': 1, 'without_mask': 2}


In [11]:
model.save("model.keras")   # NOT .h5

In [13]:
model = tf.keras.models.load_model("D:/DEBI/mask/model.keras")